# Extract landmarks using temporal linear interpolation
This notebook extracts coordinates from the Front view video files and stretches/squeezes them to exactly 60 frames using linear interpolation.

In [9]:
import os
import cv2
import mediapipe as mp
import numpy as np
from tqdm import tqdm

In [10]:
DATA_DIR = "../Datasets/Processed_Data"
SAVE_DIR = "../Datasets/Landmarks_Interpolated"
os.makedirs(SAVE_DIR, exist_ok=True)

In [11]:
mp_holistic = mp.solutions.holistic

In [12]:
IMPORTANT_FACE_IDX = [
    # Eyes
    33, 133, 159, 145, 468, 469,     # Left eye
    263, 362, 386, 374, 471, 472,    # Right eye
    
    # Eyebrows
    105, 107, 55, 65, 52,             # Left eyebrow
    285, 295, 282, 283, 336,          # Right eyebrow
    
    # Nose (bridge + tip)
    1, 2, 98, 327, 94, 97, 168, 197,
    
    # Mouth (outer + inner)
    13, 14, 78, 308, 82, 312,
    87, 317, 88, 95, 178, 191,
    80, 81, 82, 311, 310, 415,
    291, 308, 324, 318, 402, 317
]

In [13]:
print("Using important face landmarks:", len(IMPORTANT_FACE_IDX))

Using important face landmarks: 54


In [14]:
# Extract function with dynamic temporal interpolation instead of zero-padding/truncation
def interpolate_sequence(seq, target_len=60):
    T = len(seq)
    if T == 0:
        return np.zeros((target_len, 387))
    if T == target_len:
        return np.array(seq)
    
    seq = np.array(seq)
    D = seq.shape[1]
    
    orig_grid = np.linspace(0, 1, T)
    target_grid = np.linspace(0, 1, target_len)
    
    new_seq = np.zeros((target_len, D))
    for d in range(D):
        new_seq[:, d] = np.interp(target_grid, orig_grid, seq[:, d])
        
    return new_seq

def extract_raw_landmarks(video_path, max_frames=60):
    cap = cv2.VideoCapture(video_path)
    seq = []

    with mp_holistic.Holistic(
        model_complexity=1,
        refine_face_landmarks=True
    ) as holistic:

        while True:
            ret, frame = cap.read()
            if not ret:
                break

            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = holistic.process(frame)

            lm = []

            # IMPORTANT FACE LANDMARKS ONLY
            if results.face_landmarks:
                face = results.face_landmarks.landmark
                for idx in IMPORTANT_FACE_IDX:
                    p = face[idx]
                    lm += [p.x, p.y, p.z]
            else:
                lm += [0] * len(IMPORTANT_FACE_IDX) * 3

            # LEFT HAND (21)
            if results.left_hand_landmarks:
                for p in results.left_hand_landmarks.landmark:
                    lm += [p.x, p.y, p.z]
            else:
                lm += [0] * 21 * 3

            # RIGHT HAND (21)
            if results.right_hand_landmarks:
                for p in results.right_hand_landmarks.landmark:
                    lm += [p.x, p.y, p.z]
            else:
                lm += [0] * 21 * 3

            # POSE (33)
            if results.pose_landmarks:
                for p in results.pose_landmarks.landmark:
                    lm += [p.x, p.y, p.z]
            else:
                lm += [0] * 33 * 3

            seq.append(lm)

    cap.release()

    return interpolate_sequence(seq, max_frames)


In [15]:
views = ["Front"]
for view in views:
    view_path = os.path.join(DATA_DIR, view)
    out_view_path = os.path.join(SAVE_DIR, view)
    os.makedirs(out_view_path, exist_ok=True)

    word_folders = os.listdir(view_path)

    for word in tqdm(word_folders, desc=f"Extracting {view}"):
        word_input_dir = os.path.join(view_path, word)
        word_output_dir = os.path.join(out_view_path, word)
        os.makedirs(word_output_dir, exist_ok=True)

        files = [f for f in os.listdir(word_input_dir) if f.endswith(".mp4")]

        for f in files:
            video_path = os.path.join(word_input_dir, f)
            arr = extract_raw_landmarks(video_path)

            save_path = os.path.join(word_output_dir, f.replace(".mp4", ".npy"))
            np.save(save_path, arr)


Extracting Front:   0%|          | 0/401 [04:05<?, ?it/s]


KeyboardInterrupt: 